# FloorPlan-AI: U-Net (ResNet-18) fine-tuning on CubiCasa5K — Colab runbook

**Runtime: T4 GPU** (Runtime > Change runtime type). Run the cells in order.

**What this notebook does.** Real gradient-based fine-tuning of a U-Net whose encoder is an ImageNet-pretrained ResNet-18, on real
CubiCasa5K floor plans, then evaluation on held-out test plans and polygon-JSON export.
It does **not** use synthetic data, Qwen2.5-VL or any pretrained-only inference.

**Two stages with a human gate between them**
* **Stage A (data):** download the official zip, inspect the REAL SVG structure, build masks and overlays.
  Look at the overlays. Download `stageA_for_review.zip` and send it to Claude for an independent look before training.
* **Stage B (model):** only after you set `OVERLAYS_CONFIRMED = True`: tests, overfit-8 sanity run, full training, evaluation, packaging.

Nothing in this notebook has been executed by its author (the authoring sandbox has no GPU, no torch and cannot reach Zenodo).
Any failure you hit is useful information: send the cell output.

## Stage A — data

In [ ]:
# A1. Clone the branch and install light dependencies (torch/torchvision are already on Colab)
import os, subprocess, sys
REPO = "/content/FloorPlan-AI"
if not os.path.isdir(REPO):
    !git clone https://github.com/sandhiya-1234/FloorPlan-AI.git {REPO}
os.chdir(REPO)
!git checkout repair/real-vlm-pipeline
!git pull --ff-only
!git log --oneline -1
!pip install -q -r requirements-seg.txt
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# A2. (optional) keep outputs across Colab disconnects: copy results to Google Drive at the end of each stage
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/floorplan_ai"
os.makedirs(DRIVE, exist_ok=True)
print("Drive folder:", DRIVE)

In [ ]:
# A3. Download the official CubiCasa5K zip (Zenodo record 2613548) and verify the checksum shown on the record page
import hashlib
ZIP = "/content/cubicasa5k.zip"
URL = "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
EXPECTED_MD5 = "0ce0b203d1e3c125b51087b219bd23b9"   # from the Zenodo record page (file cubicasa5k.zip, 5.5 GB)
if not os.path.isfile(ZIP) or os.path.getsize(ZIP) < 5_000_000_000:
    !wget -c --tries=5 --timeout=60 -O {ZIP} "{URL}"
md5 = hashlib.md5()
with open(ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        md5.update(chunk)
print("size GB:", round(os.path.getsize(ZIP) / 1e9, 2), "| md5:", md5.hexdigest())
assert md5.hexdigest() == EXPECTED_MD5, "checksum mismatch: the download is incomplete or the file changed. Do not continue."
print("checksum OK")

In [ ]:
# A4. INSPECT the real archive: folder layout, official split lists, SVG group ids/classes, room names, selector presence.
# This exits non-zero (and the next cells must not be run) if an official selector never occurs in the real files.
os.makedirs("/content/stageA", exist_ok=True)
!python -m training.prepare_dataset inspect --zip {ZIP} --n 60 2>&1 | tee /content/stageA/inspect_report.txt

In [ ]:
# A5. Build the subset: seeded sample from the OFFICIAL train/val/test lists, masks from the explicit mapping,
# cross-split duplicate removal, overlays, manifest.  (Reads the zip directly; no LMDB, no full extraction.)
!python -m training.prepare_dataset prepare --zip {ZIP} --out data/seg --n-overlays 10 2>&1 | tee /content/stageA/prepare_report.txt

In [ ]:
# A6. LOOK at the overlays. Left: image. Right: ground-truth mask from the mapping (blue Room, red Wall, green Door, yellow Window, hatched grey ignored).
# Walls must sit on the drawn walls, doors/windows on the openings, rooms inside the walls. Misaligned masks = do NOT train.
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
files = sorted(glob.glob("data/seg/overlays/train_*.png"))[:6] + sorted(glob.glob("data/seg/overlays/val_*.png"))[:2] + sorted(glob.glob("data/seg/overlays/test_*.png"))[:2]
for f in files:
    plt.figure(figsize=(16, 7)); plt.imshow(mpimg.imread(f)); plt.axis("off"); plt.title(os.path.basename(f)); plt.show()

In [ ]:
# A7. Package everything needed for an independent review, and back it up to Drive.
!cp data/seg/manifest.json /content/stageA/manifest.json
!cd /content/stageA && zip -q -r stageA_for_review.zip inspect_report.txt prepare_report.txt manifest.json && cd {REPO}/data/seg && zip -q -r /content/stageA/overlays.zip overlays
!cp /content/stageA/*.zip {DRIVE}/
from google.colab import files
files.download("/content/stageA/stageA_for_review.zip")
files.download("/content/stageA/overlays.zip")

### Gate — do not continue until the overlays have been checked
Send `stageA_for_review.zip` and `overlays.zip` to Claude (attach them in the chat). Claude will look at the overlay images and the
inspect report and say whether the mapping is right. Then set the flag below. Training refuses to start without the confirmation file.

In [ ]:
# A8. Human confirmation gate. Change to True only after YOU looked at the overlays and they line up with the images.
OVERLAYS_CONFIRMED = False
from datetime import datetime, timezone
if OVERLAYS_CONFIRMED:
    open("data/seg/overlays_confirmed.txt", "w").write(f"overlays inspected and confirmed {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n")
    print("confirmation file written")
else:
    print("NOT confirmed: Stage B will refuse to train. Fix the mapping or ask Claude to review.")

## Stage B — model (only after the gate)

In [ ]:
# B1. Pre-flight: the full test suite, including the torch tests (shapes, loss, optimiser loop, train->checkpoint->predict->evaluate
# round trip on tiny fake data) and the shapely tests that could not be run in the authoring sandbox. Fix failures before training.
!python -m pytest tests -q -rs

In [ ]:
# B2. Pipeline sanity: overfit 8 real training images (train and validate on the SAME 8). This only shows that the code can learn.
# It is labelled overfit_sanity and is never a result. Loss should fall clearly and mIoU on these 8 images should rise.
!python training/train_segmentation.py --overfit 8 --epochs 30 --run-name overfit8

In [ ]:
# B3. FULL fine-tuning on the prepared training split, model selection on the validation split.
# ImageNet-pretrained ResNet-18 encoder + U-Net decoder, AdamW, CE + Dice, fp16 autocast, 512x512 letterboxed, batch 4.
# Saves runs/seg/unet_resnet18/{best.pt, history.json, config.json, train_summary.json}.
!python training/train_segmentation.py --run-name unet_resnet18

In [ ]:
# B4. Training curves from the saved history (real values read from runs/seg/unet_resnet18/history.json)
import json, matplotlib.pyplot as plt
h = json.load(open("runs/seg/unet_resnet18/history.json"))
ep = [r["epoch"] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [r["train_loss"] for r in h], label="train"); ax[0].plot(ep, [r["val_loss"] for r in h], label="val"); ax[0].set_title("CE + Dice loss"); ax[0].legend()
ax[1].plot(ep, [r["val_mean_iou"] for r in h]); ax[1].set_title("validation mean IoU")
plt.savefig("runs/seg/unet_resnet18/training_curves.png", dpi=150, bbox_inches="tight"); plt.show()
print(json.dumps(json.load(open("runs/seg/unet_resnet18/train_summary.json")), indent=2))

In [ ]:
# B5. Evaluate on the held-out TEST split (never used for training or model selection). Aborts if any test id was seen in training.
!python evaluation/evaluate_segmentation.py --checkpoint runs/seg/unet_resnet18/best.pt --out-dir results/segmentation 2>&1 | tee results_log.txt

In [ ]:
# B6. Look at predictions: image | ground truth | prediction (first and worst test images)
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
for f in sorted(glob.glob("results/segmentation/visuals/*.png"))[:8]:
    plt.figure(figsize=(18, 6)); plt.imshow(mpimg.imread(f)); plt.axis("off"); plt.title(os.path.basename(f)); plt.show()

In [ ]:
# B7. Package results (small files, safe for Git) and put the weights somewhere outside Git (your Drive).
import hashlib, shutil
!cp results_log.txt results/segmentation/ 2>/dev/null; cp runs/seg/unet_resnet18/training_curves.png results/segmentation/
!cd {REPO} && zip -q -r /content/segmentation_results.zip results/segmentation
shutil.copy("runs/seg/unet_resnet18/best.pt", f"{DRIVE}/unet_resnet18_best.pt")
shutil.copy("/content/segmentation_results.zip", f"{DRIVE}/segmentation_results.zip")
sha = hashlib.sha256(open("runs/seg/unet_resnet18/best.pt", "rb").read()).hexdigest()
print("weights saved to", f"{DRIVE}/unet_resnet18_best.pt", "| bytes:", os.path.getsize("runs/seg/unet_resnet18/best.pt"), "\nsha256:", sha)
from google.colab import files
files.download("/content/segmentation_results.zip")

### Send back
Attach `segmentation_results.zip` (and, if you want the model in the repo's release, tell Claude where the weights live).
Claude will re-check the numbers against the raw `metrics.json`, update the README and slides from those files only, and push.